## Set Up

In [39]:
import pandas as pd
import numpy as np
import json
import pickle
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

PROCESSED = Path("../processed")
FIGURES = Path("../figures")

df = pd.read_parquet(PROCESSED / "data_merged.parquet")

with open(PROCESSED / "missing_strategy.json", "r") as f:
    missing_strategy = json.load(f)

print(df.shape)
print(missing_strategy.keys())

(590540, 434)
dict_keys(['identity_related', 'time_delta', 'v_conditional_blocks', 'categorical_other'])


In [40]:
# Binary flag: 1 = data was present, 0 = missing 
id_cols = missing_strategy["identity_related"]
d_cols = missing_strategy["time_delta"]

for col in id_cols + d_cols:
    if col in df.columns:
        df[f"{col}_was_missing"] = df[col].isnull().astype(int)

print(f"Flag columns created: {len(id_cols) + len(d_cols)}")

Flag columns created: 34


In [41]:

v_cols = missing_strategy["v_conditional_blocks"]
v_cols_in_df = [c for c in v_cols if c in df.columns]

df["V_block_missing_count"] = df[v_cols_in_df].isnull().sum(axis=1)

print("V_block_missing_count stats:")
print(df["V_block_missing_count"].describe())
print("\nCorrelation with isFraud:", df["V_block_missing_count"].corr(df["isFraud"]).round(4))

V_block_missing_count stats:
count    590540.000000
mean        126.645973
std          59.180690
min           0.000000
25%         159.000000
50%         159.000000
75%         159.000000
max         159.000000
Name: V_block_missing_count, dtype: float64

Correlation with isFraud: -0.1091


Numerical Imputation

In [42]:
# All numeric columns except target and ID
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in ["isFraud", "TransactionID"]]

# sentinel -999 for high-missing cols (id_, D time-delta, V conditional)
cols_to_impute_sentinel = id_cols + d_cols + v_cols_in_df
cols_to_impute_sentinel = [c for c in cols_to_impute_sentinel if c in df.columns]
df[cols_to_impute_sentinel] = df[cols_to_impute_sentinel].fillna(-999)

#  median for ALL remaining numeric cols — no inplace, direct assignment
medians = df[num_cols].median()
df[num_cols] = df[num_cols].fillna(medians)


total_remaining = df[num_cols].isnull().sum().sum()
cols_still_null = [c for c in num_cols if df[c].isnull().sum() > 0]

print(f"Sentinel-imputed columns: {len(cols_to_impute_sentinel)}")
print(f"Columns still with nulls: {len(cols_still_null)}")
print(f"Total remaining nulls (numeric): {total_remaining}")

Sentinel-imputed columns: 193
Columns still with nulls: 0
Total remaining nulls (numeric): 0


Categorical Imputation

In [43]:
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

for col in cat_cols:
    df[col] = df[col].fillna("unknown")

print(f"Categorical columns imputed: {len(cat_cols)}")
print(f"Remaining nulls (categorical): {df[cat_cols].isnull().sum().sum()}")
print("\nCategorical columns:", cat_cols)

/var/folders/wt/k04sj_gj2txbf9dld5w6kv840000gn/T/ipykernel_87520/1165504387.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object"]).columns.tolist()


Categorical columns imputed: 31
Remaining nulls (categorical): 0

Categorical columns: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


Temporal Features

In [44]:
df["day"] = (df["TransactionDT"] / 86400).astype(int)
df["hour"] = ((df["TransactionDT"] % 86400) / 3600).astype(int)
df["day_of_week"] = df["day"] % 7

print("Fraud rate by hour of day (top 5):")
print(df.groupby("hour")["isFraud"].mean().sort_values(ascending=False).head(5))

Fraud rate by hour of day (top 5):
hour
7    0.106102
8    0.093014
9    0.089956
6    0.077743
5    0.070302
Name: isFraud, dtype: float64


/var/folders/wt/k04sj_gj2txbf9dld5w6kv840000gn/T/ipykernel_87520/2331345022.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["day"] = (df["TransactionDT"] / 86400).astype(int)
/var/folders/wt/k04sj_gj2txbf9dld5w6kv840000gn/T/ipykernel_87520/2331345022.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["hour"] = ((df["TransactionDT"] % 86400) / 3600).astype(int)
/var/folders/wt/k04sj_gj2txbf9dld5w6kv840000gn/T/ipykernel_87520/2331345022.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the resul

Financial Features

In [45]:
df["amt_log"] = np.log1p(df["TransactionAmt"])

card_stats = df.groupby("card1")["TransactionAmt"].agg(
    card_mean="mean", card_std="std"
).reset_index()
df = df.merge(card_stats, on="card1", how="left")

df["amt_vs_card_mean"] = df["TransactionAmt"] / (df["card_mean"] + 1)
df["amt_zscore_card"] = (df["TransactionAmt"] - df["card_mean"]) / (df["card_std"] + 1)

print("New financial features created: amt_log, amt_vs_card_mean, amt_zscore_card")
print(df[["TransactionAmt", "amt_log", "amt_vs_card_mean", "amt_zscore_card"]].describe().round(3))

New financial features created: amt_log, amt_vs_card_mean, amt_zscore_card
       TransactionAmt     amt_log  amt_vs_card_mean  amt_zscore_card
count      590540.000  590540.000        590540.000       587096.000
mean          135.027       4.383             0.990            0.000
std           239.163       0.937             1.355            0.981
min             0.251       0.224             0.003           -4.117
25%            43.321       3.791             0.379           -0.459
50%            68.769       4.245             0.629           -0.284
75%           125.000       4.836             1.088            0.097
max         31937.391      10.372           113.718           36.863


/var/folders/wt/k04sj_gj2txbf9dld5w6kv840000gn/T/ipykernel_87520/2606903701.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["amt_log"] = np.log1p(df["TransactionAmt"])
/var/folders/wt/k04sj_gj2txbf9dld5w6kv840000gn/T/ipykernel_87520/2606903701.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["amt_vs_card_mean"] = df["TransactionAmt"] / (df["card_mean"] + 1)
/var/folders/wt/k04sj_gj2txbf9dld5w6kv840000gn/T/ipykernel_87520/2606903701.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result

In [46]:
df["card_std"] = df["card_std"].astype(float).fillna(0.0)
df["amt_zscore_card"] = df["amt_zscore_card"].astype(float).fillna(0.0)

print(f"Nulls in card_std: {df['card_std'].isnull().sum()}")
print(f"Nulls in amt_zscore_card: {df['amt_zscore_card'].isnull().sum()}")
print(f"Total nulls in df: {df.isnull().sum().sum()}")

Nulls in card_std: 0
Nulls in amt_zscore_card: 0
Total nulls in df: 0


Label Encoding

In [47]:
from sklearn.preprocessing import LabelEncoder
import pickle

label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

print(f"Label encoded {len(cat_cols)} categorical columns.")

Label encoded 31 categorical columns.


In [48]:
cols_to_drop = ["TransactionID", "TransactionDT"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

total_nulls = df.isnull().sum().sum()
print(f"Total remaining nulls: {total_nulls}")
print(f"Final shape: {df.shape}")

df.to_parquet(PROCESSED / "data_processed.parquet", index=False)

with open(PROCESSED / "label_encoders.pkl", "wb") as f:
    pickle.dump(label_encoders, f)

print("Saved: data_processed.parquet, label_encoders.pkl")

Total remaining nulls: 0
Final shape: (590540, 475)
Saved: data_processed.parquet, label_encoders.pkl


## Summary

- Applied differentiated imputation based on missing-data analysis from notebook 01:
  - `id_*` (28 cols) and `D*` time-delta (6 cols): binary flag per column 
    (`_was_missing`) + sentinel value -999.
  - `V*` conditional blocks (159 cols): aggregated flag `V_block_missing_count` 
    + sentinel -999. Avoids dimensionality explosion while preserving the signal.
  - Remaining categorical columns: imputed with "unknown".
  - Remaining numeric columns (card2, card3, card5, addr, dist, D1-D15, etc.): 
    imputed with column median.
  - `card_std` and `amt_zscore_card`: imputed with 0 (single-transaction cards 
    have no computable std — z-score of 0 represents neutral deviation).
- **New features engineered:**
  - `hour`, `day_of_week` from `TransactionDT` (temporal behavior patterns)
  - `amt_log` (reduces skewness of TransactionAmt distribution)
  - `amt_vs_card_mean`: ratio of transaction amount to card's historical mean
  - `amt_zscore_card`: how many std deviations this transaction is from the 
    card's typical spend — strong fraud signal candidate
- Label encoding applied to all categorical columns. Encoders saved to 
  `label_encoders.pkl` for reuse in the Streamlit dashboard.
- Final dataset: **0 nulls, 475 columns**, ready for modeling.

**Outputs:** `data_processed.parquet`, `label_encoders.pkl`